# Google Colab Pro Training Setup for Moisture Detection Model (Simplified Structure)

## Overview

This notebook provides step-by-step instructions to train a custom image classification model using Google Colab Pro with your 23,000+ labeled images from Google Drive with simplified structure (after restructuring), then convert it for use with your existing TensorFlow.js application.

## Prerequisites

- Google Colab Pro subscription ($10/month)
- Google Drive with organized image dataset in simplified structure:
  ```
  Predicto_GPT_Taining_images/
  ├── 0/
  │   ├── image1_day.jpg
  │   ├── image2_night.jpg
  │   └── ...
  ├── 25/
  │   ├── photo_1_day.png
  │   ├── photo_2_night.png
  │   └── ...
  ├── ... (50, 75, 100, 130, 175, 200, 250, 300, 350, 400, 450)
  └── Invalid/
      ├── invalid_image1.jpg
      └── ...
  ```
- Basic understanding of Python and machine learning concepts

## Expected Results

- **Training Time**: 2-4 hours on Colab Pro GPU
- **Model Accuracy**: Typically 85-95% with proper data
- **Output**: TensorFlow.js compatible model files
- **Model Size**: 5-15MB (with quantization)

## Step 1: Setup Google Colab Pro Environment

### 1.1 Subscribe and Access

1. Go to [Google Colab](https://colab.research.google.com)
2. Subscribe to Colab Pro ($10/month) for GPU access and longer runtimes
3. Create a new notebook: **File → New notebook**

### 1.2 Configure Runtime

1. **Runtime → Change runtime type**
2. Set **Hardware accelerator** to **GPU**
3. Set **Runtime shape** to **High-RAM** (if available)
4. Click **Save**

### 1.3 Verify GPU Access

In [1]:
# Check GPU availability
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

# Get GPU details
if tf.config.list_physical_devices('GPU'):
    gpu = tf.config.experimental.get_device_details(tf.config.list_physical_devices('GPU')[0])
    print("GPU details:", gpu)
else:
    print("⚠️  No GPU detected - check runtime settings")

TensorFlow version: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU details: {'compute_capability': (8, 0), 'device_name': 'NVIDIA A100-SXM4-40GB'}


## Step 2: Mount Google Drive and Install Dependencies

### 2.1 Mount Google Drive

In [2]:
from google.colab import drive
import os
import time

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)
time.sleep(3)  # Wait for sync

# Verify mount
print("✅ Google Drive mounted successfully")
print("Available folders:", os.listdir('/content/drive/MyDrive/Predicto_GPT_Taining_images/')[:14])

Mounted at /content/drive
✅ Google Drive mounted successfully
Available folders: ['50', '25', '200', '175', '350', '400', '300', '250', '450', '75', '0', '130', '100', 'Invalid']


### 2.1.5 Package Installation Notes

**Note**: This notebook uses proven compatible versions and streamlined installation to run continuously without requiring runtime restarts.

### 2.2 Install Required Packages

**Note**: This approach uses tested compatible versions to avoid common dependency conflicts and runs seamlessly without requiring runtime restarts.

In [3]:
# Install packages with proven compatibility for Google Colab
print("📦 Installing packages with reliable versions for Google Colab...")

# First, completely restart the session and install fresh packages
!pip install --upgrade pip setuptools wheel -q

# Uninstall potentially problematic packages completely
!pip uninstall -y scikit-learn scipy numpy -q 2>/dev/null || true

# Install numpy first with specific version
!pip install "numpy==1.24.3" --force-reinstall --no-deps -q

# Install scipy before sklearn to avoid dependency issues
!pip install "scipy==1.11.4" -q

# Install sklearn with compatible version and allow it to install its dependencies
!pip install "scikit-learn==1.3.2" -q

# Install other required packages
!pip install tensorflowjs matplotlib pillow pandas seaborn joblib threadpoolctl -q

print("✅ All packages installed successfully!")
print("🔄 Package installation complete - ready to continue!")

📦 Installing packages with reliable versions for Google Colab...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 81.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a probl

In [4]:
# Import essential libraries
print("🔍 Importing essential libraries...")
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")

# Verify compatibility
if hasattr(tf, 'config') and tf.config.list_physical_devices('GPU'):
    print("✅ GPU detected and accessible")
else:
    print("⚠️ No GPU detected - check runtime settings")

# Handle seaborn import with fallback
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
    print("✅ Seaborn imported successfully")
except ImportError as e:
    print(f"⚠️ Seaborn import failed: {e}")
    print("Will use matplotlib for all visualizations")
    SEABORN_AVAILABLE = False

from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
import tensorflowjs as tfjs
from datetime import datetime
import json
from collections import Counter
import os
import shutil
import glob
from pathlib import Path

# Import sklearn functions
try:
    from sklearn.model_selection import train_test_split
    from sklearn.utils.class_weight import compute_class_weight
    SKLEARN_AVAILABLE = True
    print("✅ Scikit-learn imported successfully")
except ImportError as e:
    print(f"⚠️ Scikit-learn import failed: {e}")
    print("Using built-in alternatives for train_test_split and class_weight")
    SKLEARN_AVAILABLE = False

# Fallback functions if sklearn is not available
if not SKLEARN_AVAILABLE:
    def train_test_split(*arrays, test_size=None, train_size=None, stratify=None, random_state=None):
        """Simple fallback for train_test_split with stratification"""
        if random_state:
            np.random.seed(random_state)

        main_array = arrays[0]
        n_samples = len(main_array)

        if test_size is None:
            test_size = 0.25
        if test_size < 1:
            test_size = int(test_size * n_samples)

        indices = np.arange(n_samples)

        if stratify is not None:
            unique_labels = np.unique(stratify)
            train_indices = []
            test_indices = []

            for label in unique_labels:
                label_indices = indices[stratify == label]
                np.random.shuffle(label_indices)
                label_test_size = int(len(label_indices) * (test_size / n_samples))
                test_indices.extend(label_indices[:label_test_size])
                train_indices.extend(label_indices[label_test_size:])
        else:
            np.random.shuffle(indices)
            test_indices = indices[:test_size]
            train_indices = indices[test_size:]

        result = []
        for array in arrays:
            array = np.array(array)
            result.extend([array[train_indices], array[test_indices]])

        return result

    def compute_class_weight(class_weight, classes, y):
        """Simple fallback for compute_class_weight"""
        if class_weight == 'balanced':
            unique_classes, counts = np.unique(y, return_counts=True)
            total_samples = len(y)
            n_classes = len(unique_classes)
            weights = total_samples / (n_classes * counts)
            return weights
        else:
            return np.ones(len(classes))

print("✅ All packages imported successfully")

🔍 Importing essential libraries...
TensorFlow version: 2.19.0
NumPy version: 2.0.2
✅ GPU detected and accessible


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 3: Data Loading for Simplified Structure

In [ ]:
# Configure paths (UPDATE THESE TO MATCH YOUR GOOGLE DRIVE STRUCTURE)
DRIVE_DATA_PATH = '/content/drive/MyDrive/Predicto_GPT_Taining_images'
MODEL_NAME = 'iron-custom-test-v1.0'  # Test version
WORK_DIR = '/content/moisture_detection_training'

# Data split configuration
TRAIN_SPLIT = 0.7   # 70% for training
VAL_SPLIT = 0.2     # 20% for validation
TEST_SPLIT = 0.1    # 10% for testing

# 🚀 QUICK TEST CONFIGURATION - For fast end-to-end testing
MAX_IMAGES_PER_CLASS = 5  # ⚠️ VERY LIMITED for quick testing!
QUICK_TEST_MODE = True    # Enable quick test mode

# Define your 14 classes
CLASS_NAMES = ['0', '25', '50', '75', '100', '130', '175', '200', '250', '300', '350', '400', '450', 'Invalid']

# Create working directory
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

print("🧪 QUICK TEST MODE ENABLED")
print("=" * 50)
print("⚠️  This configuration is for fast end-to-end testing only!")
print("⚠️  Model accuracy will be poor due to limited data.")
print("⚠️  Use this to test the full pipeline, then increase data for real training.")
print("=" * 50)
print()
print(f"Working directory: {WORK_DIR}")
print(f"Data path: {DRIVE_DATA_PATH}")
print(f"Classes: {CLASS_NAMES}")
print(f"Data splits: Train={TRAIN_SPLIT}, Val={VAL_SPLIT}, Test={TEST_SPLIT}")
print(f"Max images per class: {MAX_IMAGES_PER_CLASS} (TEST MODE)")
print()
print("🎯 Expected completion time: ~10-15 minutes")

### 3.1 Set Data Paths and Configuration

In [ ]:
def collect_image_paths_and_labels(data_path, class_names, max_images_per_class=None):
    """
    Collect image paths and their corresponding labels from simplified structure.
    All images are directly in class folders with _day/_night suffixes.

    Args:
        data_path: Path to the root data directory
        class_names: List of class names
        max_images_per_class: Maximum number of images to collect per class (None for all)
    """
    image_paths = []
    labels = []
    class_counts = {}

    print("🔍 Scanning simplified dataset structure...")
    if max_images_per_class:
        print(f"🎯 Limiting to {max_images_per_class} images per class")

    for class_idx, class_name in enumerate(class_names):
        class_path = os.path.join(data_path, class_name)

        if not os.path.exists(class_path):
            print(f"⚠️  Warning: Class folder '{class_name}' not found at {class_path}")
            class_counts[class_name] = 0
            continue

        class_image_count = 0
        class_image_paths = []  # Collect all paths for this class first

        # Get all image files directly from class folder
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tiff']:
            pattern = os.path.join(class_path, ext)
            found_files = glob.glob(pattern)
            class_image_paths.extend(found_files)

        # Shuffle and limit the images for this class if max_images_per_class is set
        if class_image_paths:
            # Shuffle to get random selection
            np.random.shuffle(class_image_paths)

            # Limit to max_images_per_class if specified
            if max_images_per_class:
                class_image_paths = class_image_paths[:max_images_per_class]

            # Add to final lists
            for img_path in class_image_paths:
                image_paths.append(img_path)
                labels.append(class_idx)
                class_image_count += 1

        class_counts[class_name] = class_image_count
        total_available = len(glob.glob(os.path.join(class_path, '*.*'))) if os.path.exists(class_path) else 0

        if max_images_per_class and total_available > max_images_per_class:
            print(f"  {class_name:>10}: {class_image_count:>6,} images (limited from {total_available:,} available)")
        else:
            print(f"  {class_name:>10}: {class_image_count:>6,} images")

        # Show day/night distribution for this class
        day_count = len([p for p in class_image_paths if '_day.' in os.path.basename(p)])
        night_count = len([p for p in class_image_paths if '_night.' in os.path.basename(p)])
        other_count = class_image_count - day_count - night_count

        if day_count > 0 or night_count > 0:
            print(f"    └─ Day: {day_count}, Night: {night_count}" + (f", Other: {other_count}" if other_count > 0 else ""))

    return image_paths, labels, class_counts

# Set random seed for reproducible sampling
np.random.seed(42)

# Collect all image paths and labels
print("📊 Collecting image paths from simplified structure...")
all_image_paths, all_labels, class_counts = collect_image_paths_and_labels(
    DRIVE_DATA_PATH, CLASS_NAMES, MAX_IMAGES_PER_CLASS
)

total_images = len(all_image_paths)
print(f"\n✅ Dataset collection complete!")
print(f"Total images: {total_images:,}")
print(f"Total classes: {len(CLASS_NAMES)}")
if MAX_IMAGES_PER_CLASS:
    expected_max = len(CLASS_NAMES) * MAX_IMAGES_PER_CLASS
    print(f"Expected maximum (if all classes full): {expected_max:,} images")

# Verify we have data for all classes
empty_classes = [name for name, count in class_counts.items() if count == 0]
if empty_classes:
    print(f"⚠️  Empty classes found: {empty_classes}")
else:
    print("✅ All classes have images")

### 3.3 Dataset Analysis and Visualization

In [ ]:
def analyze_dataset(image_paths, labels, class_names, class_counts):
    """Basic dataset analysis without visualization"""

    # Convert to numpy arrays for easier manipulation
    labels_array = np.array(labels)

    print(f"\n📊 Dataset Analysis:")
    print("=" * 60)
    print(f"Total images: {len(image_paths):,}")
    print(f"Total classes: {len(class_names)}")

    # Class distribution
    print(f"\n📋 Class distribution:")
    for i, class_name in enumerate(class_names):
        count = class_counts[class_name]
        percentage = (count / len(image_paths)) * 100 if len(image_paths) > 0 else 0
        print(f"  {class_name:>10}: {count:>6,} images ({percentage:>5.1f}%)")

    # Check for class imbalance
    counts = list(class_counts.values())
    non_zero_counts = [c for c in counts if c > 0]

    if non_zero_counts:
        min_count = min(non_zero_counts)
        max_count = max(non_zero_counts)
        imbalance_ratio = max_count / min_count if min_count > 0 else float('inf')

        print(f"\n⚖️  Class balance analysis:")
        print(f"Min class size: {min_count:,}")
        print(f"Max class size: {max_count:,}")
        print(f"Imbalance ratio: {imbalance_ratio:.2f}:1")

        if imbalance_ratio > 5:
            print("⚠️  Significant class imbalance detected!")
            print("Consider using class weights during training.")

    return labels_array

# Analyze the collected dataset
labels_array = analyze_dataset(all_image_paths, all_labels, CLASS_NAMES, class_counts)

### 3.4 Stratified Train/Validation/Test Split

In [ ]:
def create_stratified_splits(image_paths, labels, class_names, train_split=0.7, val_split=0.2, test_split=0.1):
    """
    Create stratified train/validation/test splits ensuring each class is represented
    in all splits according to its distribution in the original dataset.
    """

    print("🔄 Creating stratified train/validation/test splits...")

    # Verify splits sum to 1
    total_split = train_split + val_split + test_split
    if abs(total_split - 1.0) > 0.001:
        raise ValueError(f"Splits must sum to 1.0, got {total_split}")

    # Convert to numpy arrays
    image_paths = np.array(image_paths)
    labels = np.array(labels)

    # First split: separate train from temp (val + test)
    train_paths, temp_paths, train_labels, temp_labels = train_test_split(
        image_paths, labels,
        test_size=(val_split + test_split),
        stratify=labels,
        random_state=42
    )

    # Second split: separate validation from test
    # Calculate relative sizes for val and test from the temp set
    val_relative_size = val_split / (val_split + test_split)

    val_paths, test_paths, val_labels, test_labels = train_test_split(
        temp_paths, temp_labels,
        test_size=(1 - val_relative_size),
        stratify=temp_labels,
        random_state=42
    )

    # Verify splits
    total_samples = len(image_paths)

    print(f"\n✅ Split creation complete:")
    print(f"Training samples:   {len(train_paths):>6,} ({len(train_paths)/total_samples*100:.1f}%)")
    print(f"Validation samples: {len(val_paths):>6,} ({len(val_paths)/total_samples*100:.1f}%)")
    print(f"Test samples:       {len(test_paths):>6,} ({len(test_paths)/total_samples*100:.1f}%)")
    print(f"Total samples:      {total_samples:>6,}")

    # Verify stratification worked
    print(f"\n📊 Per-class distribution verification:")
    for i, class_name in enumerate(class_names):
        train_count = np.sum(train_labels == i)
        val_count = np.sum(val_labels == i)
        test_count = np.sum(test_labels == i)
        total_count = train_count + val_count + test_count

        if total_count > 0:
            print(f"  {class_name:>10}: Train={train_count:>4} ({train_count/total_count*100:>4.1f}%) "
                  f"Val={val_count:>4} ({val_count/total_count*100:>4.1f}%) "
                  f"Test={test_count:>4} ({test_count/total_count*100:>4.1f}%)")

    return (train_paths, train_labels), (val_paths, val_labels), (test_paths, test_labels)

# Create the splits
(train_paths, train_labels), (val_paths, val_labels), (test_paths, test_labels) = create_stratified_splits(
    all_image_paths, all_labels, CLASS_NAMES, TRAIN_SPLIT, VAL_SPLIT, TEST_SPLIT
)

## Step 4: Custom Data Pipeline with TensorFlow Dataset

### 4.1 Image Loading and Preprocessing Functions

In [ ]:
# Create datasets with test-friendly batch size
BATCH_SIZE = 8 if QUICK_TEST_MODE else 32  # Smaller batches for limited data

# Test with a small sample first to verify everything works
print("🧪 Testing dataset creation with minimal samples...")
test_paths_small = train_paths[:3] if len(train_paths) >= 3 else train_paths
test_labels_small = train_labels[:3] if len(train_labels) >= 3 else train_labels

if len(test_paths_small) > 0:
    test_dataset_small = create_dataset(
        test_paths_small, test_labels_small, CLASS_NAMES,
        batch_size=2, shuffle=False, augment=False
    )
    print("✅ Small test successful!")
else:
    print("⚠️ No training data found - check your data path!")

# Now create full datasets
print(f"\n📊 Creating datasets (batch_size={BATCH_SIZE})...")
import time

if len(train_paths) > 0:
    start_time = time.time()
    train_dataset = create_dataset(
        train_paths, train_labels, CLASS_NAMES,
        batch_size=BATCH_SIZE, shuffle=True, augment=not QUICK_TEST_MODE  # No augmentation in test mode
    )
    train_time = time.time() - start_time
    print(f"⏱️  Training dataset created in {train_time:.1f} seconds")

    start_time = time.time()
    val_dataset = create_dataset(
        val_paths, val_labels, CLASS_NAMES,
        batch_size=BATCH_SIZE, shuffle=False, augment=False
    )
    val_time = time.time() - start_time
    print(f"⏱️  Validation dataset created in {val_time:.1f} seconds")

    start_time = time.time()
    test_dataset = create_dataset(
        test_paths, test_labels, CLASS_NAMES,
        batch_size=BATCH_SIZE, shuffle=False, augment=False
    )
    test_time = time.time() - start_time
    print(f"⏱️  Test dataset created in {test_time:.1f} seconds")

    print("✅ All datasets created successfully!")

    # Get dataset sizes
    print("📏 Calculating dataset sizes...")
    train_batches = tf.data.experimental.cardinality(train_dataset).numpy()
    val_batches = tf.data.experimental.cardinality(val_dataset).numpy()
    test_batches = tf.data.experimental.cardinality(test_dataset).numpy()

    print(f"Training batches: {train_batches}")
    print(f"Validation batches: {val_batches}")
    print(f"Test batches: {test_batches}")
else:
    print("❌ No training data available - cannot proceed with training")
    print("Please check your DRIVE_DATA_PATH and ensure images are present")

### 4.2 Dataset Visualization

In [ ]:
def load_and_preprocess_image(image_path, target_size=(224, 224)):
    """Load and preprocess a single image"""
    # Load image
    image = tf.io.read_file(image_path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image = tf.cast(image, tf.float32)

    # Ensure image has correct shape
    image.set_shape([None, None, 3])

    # Resize to target size
    image = tf.image.resize(image, target_size)

    # Normalize to [0, 1]
    image = image / 255.0

    return image

def augment_image(image, label):
    """Apply data augmentation to training images"""
    # Random rotation
    image = tf.image.rot90(image, tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32))

    # Random flip
    image = tf.image.random_flip_left_right(image)

    # Random brightness
    image = tf.image.random_brightness(image, max_delta=0.2)

    # Random contrast
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)

    # Random zoom (crop and resize)
    shape = tf.shape(image)
    crop_size = tf.random.uniform([], 0.8, 1.0) * tf.cast(tf.minimum(shape[0], shape[1]), tf.float32)
    crop_size = tf.cast(crop_size, tf.int32)

    image = tf.image.random_crop(image, [crop_size, crop_size, 3])
    image = tf.image.resize(image, [224, 224])

    # Ensure values are still in [0, 1] range
    image = tf.clip_by_value(image, 0.0, 1.0)

    return image, label

def create_dataset(image_paths, labels, class_names, batch_size=32, shuffle=True, augment=False):
    """Create a TensorFlow dataset from image paths and labels"""

    print(f"🔄 Creating dataset with {len(image_paths):,} images...")

    # Convert to tensors
    path_tensor = tf.constant(image_paths)
    label_tensor = tf.constant(labels)

    # Create dataset from tensor slices
    dataset = tf.data.Dataset.from_tensor_slices((path_tensor, label_tensor))

    # Shuffle if requested
    if shuffle:
        print("🔀 Shuffling dataset...")
        dataset = dataset.shuffle(buffer_size=len(image_paths))

    # Map image loading function
    print("📸 Loading and preprocessing images...")
    dataset = dataset.map(
        lambda path, label: (load_and_preprocess_image(path), tf.one_hot(label, len(class_names))),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    # Apply augmentation if requested
    if augment:
        print("🎨 Applying data augmentation...")
        dataset = dataset.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)

    # Filter out any problematic samples
    dataset = dataset.filter(lambda image, label: tf.reduce_all(tf.math.is_finite(image)))

    # Batch and prefetch
    print(f"📦 Batching (batch_size={batch_size}) and prefetching...")
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    print("✅ Dataset creation completed!")
    return dataset

print("🔄 Creating TensorFlow datasets...")

# Create datasets with test-friendly batch size
BATCH_SIZE = 8 if QUICK_TEST_MODE else 32  # Smaller batches for limited data

# Test with a small sample first to verify everything works
print("🧪 Testing dataset creation with minimal samples...")
test_paths_small = train_paths[:3] if len(train_paths) >= 3 else train_paths
test_labels_small = train_labels[:3] if len(train_labels) >= 3 else train_labels

if len(test_paths_small) > 0:
    test_dataset_small = create_dataset(
        test_paths_small, test_labels_small, CLASS_NAMES,
        batch_size=2, shuffle=False, augment=False
    )
    print("✅ Small test successful!")
else:
    print("⚠️ No training data found - check your data path!")

# Now create full datasets
print(f"\n📊 Creating datasets (batch_size={BATCH_SIZE})...")
import time

if len(train_paths) > 0:
    start_time = time.time()
    train_dataset = create_dataset(
        train_paths, train_labels, CLASS_NAMES,
        batch_size=BATCH_SIZE, shuffle=True, augment=not QUICK_TEST_MODE  # No augmentation in test mode
    )
    train_time = time.time() - start_time
    print(f"⏱️  Training dataset created in {train_time:.1f} seconds")

    start_time = time.time()
    val_dataset = create_dataset(
        val_paths, val_labels, CLASS_NAMES,
        batch_size=BATCH_SIZE, shuffle=False, augment=False
    )
    val_time = time.time() - start_time
    print(f"⏱️  Validation dataset created in {val_time:.1f} seconds")

    start_time = time.time()
    test_dataset = create_dataset(
        test_paths, test_labels, CLASS_NAMES,
        batch_size=BATCH_SIZE, shuffle=False, augment=False
    )
    test_time = time.time() - start_time
    print(f"⏱️  Test dataset created in {test_time:.1f} seconds")

    print("✅ All datasets created successfully!")

    # Get dataset sizes
    print("📏 Calculating dataset sizes...")
    train_batches = tf.data.experimental.cardinality(train_dataset).numpy()
    val_batches = tf.data.experimental.cardinality(val_dataset).numpy()
    test_batches = tf.data.experimental.cardinality(test_dataset).numpy()

    print(f"Training batches: {train_batches}")
    print(f"Validation batches: {val_batches}")
    print(f"Test batches: {test_batches}")
else:
    print("❌ No training data available - cannot proceed with training")
    print("Please check your DRIVE_DATA_PATH and ensure images are present")

### 4.3 Calculate Class Weights for Imbalanced Data

In [ ]:
def calculate_class_weights(labels, class_names):
    """Calculate class weights to handle imbalanced dataset"""

    # Calculate class weights
    unique_labels = np.unique(labels)
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=unique_labels,
        y=labels
    )

    # Create class weight dictionary
    class_weight_dict = {i: weight for i, weight in zip(unique_labels, class_weights)}

    print("⚖️  Class weights for handling imbalance:")
    for i, class_name in enumerate(class_names):
        if i in class_weight_dict:
            print(f"  {class_name:>10}: {class_weight_dict[i]:.3f}")

    return class_weight_dict

# Calculate class weights
class_weights = calculate_class_weights(train_labels, CLASS_NAMES)

## Step 5: Build Advanced Model Architecture

### 5.1 Create Model

In [ ]:
def create_model(num_classes, input_shape=(224, 224, 3)):
    """Create MobileNetV2-based transfer learning model"""

    # Load pre-trained MobileNetV2
    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape,
        alpha=1.0  # Width multiplier
    )

    # Freeze base model initially
    base_model.trainable = False

    # Add custom classification head with explicit input specification for TensorFlow.js compatibility
    inputs = tf.keras.Input(shape=input_shape, batch_size=None, name='input_layer')
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D(name='global_avg_pool')(x)
    x = Dropout(0.2, name='dropout_1')(x)
    x = Dense(128, activation='relu', name='dense_intermediate')(x)
    x = Dropout(0.5, name='dropout_2')(x)
    outputs = Dense(num_classes, activation='softmax', name='predictions')(x)

    model = Model(inputs=inputs, outputs=outputs, name='iron_custom_model')
    return model, base_model

# Create model
print("🏗️  Building model architecture...")
model, base_model = create_model(len(CLASS_NAMES))

# Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top_3_accuracy')]
)

print("✅ Model created and compiled")
print(f"Total parameters: {model.count_params():,}")
print(f"Trainable parameters: {sum([tf.reduce_prod(var.shape) for var in model.trainable_variables]):,}")

# Display model summary
model.summary()

## Step 6: Advanced Training with Callbacks

### 6.1 Setup Training Callbacks

In [ ]:
# Create callbacks for training optimization
callbacks = [
    # Early stopping to prevent overfitting
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),

    # Reduce learning rate when plateau
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-7,
        verbose=1
    ),

    # Save best model
    tf.keras.callbacks.ModelCheckpoint(
        'best_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=False,
        verbose=1
    ),

    # CSV logger for training history
    tf.keras.callbacks.CSVLogger('training_log.csv'),
]

print("✅ Training callbacks configured")

### 6.2 Phase 1: Train with Frozen Base

In [ ]:
print("\n🚀 Phase 1: Quick training with frozen base model...")
print("=" * 60)

# Quick training configuration for testing
PHASE1_EPOCHS = 3 if QUICK_TEST_MODE else 25
PHASE2_EPOCHS = 5 if QUICK_TEST_MODE else 40

print(f"Training epochs: Phase 1 = {PHASE1_EPOCHS}, Phase 2 = {PHASE2_EPOCHS}")
if QUICK_TEST_MODE:
    print("⚠️ QUICK TEST MODE: Using minimal epochs for fast completion!")

# Record start time
import time
phase1_start = time.time()

# Train with frozen base model
history1 = model.fit(
    train_dataset,
    epochs=PHASE1_EPOCHS,
    validation_data=val_dataset,
    callbacks=callbacks if not QUICK_TEST_MODE else [],  # Skip callbacks in test mode
    class_weight=class_weights if not QUICK_TEST_MODE else None,  # Skip class weights in test mode
    verbose=1
)

phase1_duration = time.time() - phase1_start
print(f"\n⏱️  Phase 1 completed in {phase1_duration/60:.1f} minutes")

### 6.3 Phase 2: Fine-tuning with Unfrozen Base

In [ ]:
print("\n🔥 Phase 2: Quick fine-tuning with unfrozen base model...")
print("=" * 60)

# Unfreeze base model for fine-tuning
base_model.trainable = True

# Compile with lower learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),  # Lower LR
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top_3_accuracy')]
)

print(f"Trainable parameters after unfreezing: {sum([tf.reduce_prod(var.shape) for var in model.trainable_variables]):,}")

# Record start time
phase2_start = time.time()

# Fine-tune the model
history2 = model.fit(
    train_dataset,
    epochs=PHASE2_EPOCHS,
    validation_data=val_dataset,
    callbacks=callbacks if not QUICK_TEST_MODE else [],  # Skip callbacks in test mode
    class_weight=class_weights if not QUICK_TEST_MODE else None,  # Skip class weights in test mode
    verbose=1
)

phase2_duration = time.time() - phase2_start
total_training_time = phase1_duration + phase2_duration

print(f"\n⏱️  Phase 2 completed in {phase2_duration/60:.1f} minutes")
print(f"🎯 Total training time: {total_training_time/60:.1f} minutes")

if QUICK_TEST_MODE:
    print(f"\n🧪 QUICK TEST COMPLETE!")
    print(f"✅ Full pipeline completed in {total_training_time/60:.1f} minutes")
    print("⚠️  Model accuracy will be poor - this is for testing the pipeline only!")

## Step 7: Model Evaluation and Testing

### 7.1 Load Best Model and Evaluate

In [ ]:
# Use current model for evaluation
best_model = model

# Quick evaluation to get basic metrics
print("🧪 Quick model evaluation...")

# Get basic performance metrics
val_batches = tf.data.experimental.cardinality(val_dataset).numpy()
if val_batches > 0:
    val_results = best_model.evaluate(val_dataset, verbose=0)
    val_loss, val_accuracy, val_top3_acc = val_results
    print(f"Validation Accuracy: {val_accuracy:.4f} ({val_accuracy*100:.2f}%)")
else:
    val_loss, val_accuracy, val_top3_acc = 0.0, 0.0, 0.0
    print("⚠️ No validation data available")

print(f"✅ Training completed - proceeding to model conversion")

### 7.2 Skipping Detailed Analysis

*Note: Visualization and detailed analysis sections removed to focus on TensorFlow.js model generation only.*

In [ ]:
# Try importing sklearn metrics - use fallback if not available
try:
    # Restart and reimport if there are issues
    import importlib
    import sys
    if 'sklearn' in sys.modules:
        # Clear sklearn from cache if it's causing issues
        modules_to_remove = [key for key in sys.modules.keys() if key.startswith('sklearn')]
        for module in modules_to_remove:
            del sys.modules[module]

    from sklearn.metrics import classification_report, confusion_matrix
    SKLEARN_METRICS_AVAILABLE = True
    print("✅ sklearn metrics imported successfully")

    # Quick test to ensure it works
    test_y_true = [0, 1, 2, 0, 1, 2]
    test_y_pred = [0, 2, 1, 0, 0, 1]
    test_report = classification_report(test_y_true, test_y_pred, output_dict=True)
    print("✅ sklearn metrics test passed")

except Exception as e:
    print(f"⚠️ sklearn.metrics not available: {e}")
    print("Using simplified alternatives for classification analysis")
    SKLEARN_METRICS_AVAILABLE = False

    def classification_report(y_true, y_pred, target_names=None, output_dict=False):
        """Simple fallback for classification report"""
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)

        # Calculate basic metrics per class
        classes = np.unique(np.concatenate([y_true, y_pred]))
        report_dict = {}

        for cls in classes:
            true_positives = np.sum((y_true == cls) & (y_pred == cls))
            false_positives = np.sum((y_true != cls) & (y_pred == cls))
            false_negatives = np.sum((y_true == cls) & (y_pred != cls))

            precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
            recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
            f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
            support = np.sum(y_true == cls)

            class_name = target_names[cls] if target_names else str(cls)
            report_dict[class_name] = {
                'precision': precision,
                'recall': recall,
                'f1-score': f1_score,
                'support': support
            }

        # Overall accuracy
        accuracy = np.mean(y_true == y_pred)
        report_dict['accuracy'] = accuracy

        if output_dict:
            return report_dict
        else:
            # Simple text output
            text_report = "Classification Report (Simplified):\n"
            text_report += f"Accuracy: {accuracy:.4f}\n"
            for class_name, metrics in report_dict.items():
                if class_name != 'accuracy':
                    text_report += f"{class_name}: P={metrics['precision']:.3f} R={metrics['recall']:.3f} F1={metrics['f1-score']:.3f} ({metrics['support']} samples)\n"
            return text_report

    def confusion_matrix(y_true, y_pred, labels=None):
        """Simple fallback for confusion matrix"""
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)

        if labels is None:
            labels = np.unique(np.concatenate([y_true, y_pred]))

        matrix = np.zeros((len(labels), len(labels)), dtype=int)

        for i, true_label in enumerate(labels):
            for j, pred_label in enumerate(labels):
                matrix[i, j] = np.sum((y_true == true_label) & (y_pred == pred_label))

        return matrix

# If we can run detailed analysis, do it - otherwise skip
if SKLEARN_METRICS_AVAILABLE and not QUICK_TEST_MODE:
    print("📋 sklearn metrics available - can run detailed analysis")
else:
    print("📋 Skipping detailed classification analysis...")
    print("🎯 Focus: Generate TensorFlow.js model files only")

print("✅ Proceeding to model conversion...")

### 7.3 Skipping Confusion Matrix

*Note: Confusion matrix visualization removed to focus on TensorFlow.js model generation only.*

In [ ]:
# Skip visualization sections to focus on model conversion
print("📊 Skipping detailed visualization sections...")
print("🎯 Focus: Generate TensorFlow.js model files only")
print("✅ Proceeding directly to TensorFlow.js conversion...")

# If you need to run detailed analysis later with working sklearn, you can do:
# 1. Restart the runtime completely (Runtime -> Restart runtime)
# 2. Run the cells again with the improved package installation
# 3. The fallback functions above will work for basic analysis

## Step 8: Convert to TensorFlow.js Format

### 8.1 Convert and Save Model

In [ ]:
# Convert model to TensorFlow.js format
print("🔄 Converting model to TensorFlow.js format...")

# Create output directory in Google Drive
tfjs_output_path = f'/content/drive/MyDrive/{MODEL_NAME}_tfjs'
os.makedirs(tfjs_output_path, exist_ok=True)

# Convert to TensorFlow.js format with proper configuration
try:
    tfjs.converters.save_keras_model(
        best_model,
        tfjs_output_path,
        quantization_bytes=2,  # Use 16-bit quantization for smaller size
        metadata={'name': MODEL_NAME, 'version': '2.0.0'},
        save_traces=False  # Helps with InputLayer compatibility
    )
    print("✅ TensorFlow.js conversion successful!")

    # Verify the model.json file was created properly
    model_json_path = os.path.join(tfjs_output_path, 'model.json')
    if os.path.exists(model_json_path):
        print("✅ model.json file verified")
    else:
        raise Exception("model.json file not found after conversion")

except Exception as conversion_error:
    print(f"❌ Initial conversion failed: {conversion_error}")
    print("🔄 Attempting alternative conversion approach...")

    # Alternative approach: Create a clean model for conversion
    try:
        # Create new input with explicit specification
        clean_input = tf.keras.Input(
            shape=(224, 224, 3),
            batch_size=None,
            dtype=tf.float32,
            name='input_1'
        )

        # Get the model's prediction using the clean input
        clean_output = best_model(clean_input)

        # Create a new clean model
        clean_model = tf.keras.Model(
            inputs=clean_input,
            outputs=clean_output,
            name='clean_iron_model'
        )

        # Convert the clean model
        tfjs.converters.save_keras_model(
            clean_model,
            tfjs_output_path,
            quantization_bytes=2,
            metadata={'name': MODEL_NAME, 'version': '2.0.0'},
            save_traces=False
        )

        print("✅ Alternative conversion approach successful!")

    except Exception as alt_error:
        print(f"❌ Alternative conversion also failed: {alt_error}")
        print("⚠️  Manual intervention may be required for model conversion")

print(f"✅ Model converted and saved to: {tfjs_output_path}")

# Check output files
output_files = os.listdir(tfjs_output_path)
print(f"\n📁 Generated files:")
for file in output_files:
    file_path = os.path.join(tfjs_output_path, file)
    file_size = os.path.getsize(file_path)
    print(f"  {file}: {file_size/1024/1024:.2f} MB")

total_size = sum(os.path.getsize(os.path.join(tfjs_output_path, f))
                for f in output_files)
print(f"\n💾 Total model size: {total_size/1024/1024:.2f} MB")

### 8.2 Create Metadata File

In [ ]:
# Generate essential metadata for your application
metadata = {
    "modelInfo": {
        "name": MODEL_NAME,
        "version": "2.0.0",
        "description": f"Custom TensorFlow/Keras model trained on {total_images:,} images",
        "modelType": "Custom TensorFlow/Keras Image Classification",
        "architecture": "MobileNetV2 + Custom Head",
        "trainingDate": datetime.now().strftime("%Y-%m-%d"),
        "trainingDuration": f"{total_training_time/3600:.1f} hours"
    },
    "performance": {
        "valAccuracy": f"{val_accuracy:.4f}",
        "valAccuracyPercent": f"{val_accuracy*100:.2f}%",
        "trainingSamples": len(train_paths),
        "validationSamples": len(val_paths),
        "totalSamples": total_images
    },
    "classes": CLASS_NAMES,
    "classIndices": {class_name: i for i, class_name in enumerate(CLASS_NAMES)},
    "modelSpecs": {
        "inputShape": [224, 224, 3],
        "outputShape": [len(CLASS_NAMES)],
        "totalParameters": int(best_model.count_params()),
        "modelSizeMB": round(total_size/1024/1024, 2),
        "quantizationBits": 16
    }
}

# Save metadata
metadata_path = os.path.join(tfjs_output_path, 'metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print("✅ Metadata file created")
print("📋 Essential model information saved")

## Step 9: Generate Configuration and Instructions

### 9.1 Create Model Configuration

In [ ]:
# Generate configuration that matches your existing models.json format
model_config_entry = {
    MODEL_NAME: {
        "name": MODEL_NAME,
        "description": f"Custom TensorFlow/Keras model trained on {metadata['performance']['totalSamples']} images",
        "version": "2.0.0",
        "trainingDate": metadata['modelInfo']['trainingDate'],
        "modelType": "Custom TensorFlow/Keras Image Classification",
        "classes": CLASS_NAMES,
        "urls": {
            "model": f"https://your-hosting-url.com/{MODEL_NAME}/model.json",  # ⚠️ UPDATE THIS
            "metadata": f"https://your-hosting-url.com/{MODEL_NAME}/metadata.json"  # ⚠️ UPDATE THIS
        },
        "performance": {
            "valAccuracy": metadata['performance']['valAccuracyPercent'],
            "trainingSamples": metadata['performance']['trainingSamples'],
            "validationSamples": metadata['performance']['validationSamples']
        }
    }
}

# Save configuration
config_path = f'/content/drive/MyDrive/{MODEL_NAME}_config.json'
with open(config_path, 'w') as f:
    json.dump(model_config_entry, f, indent=2)

print("✅ Model configuration generated for your codebase")
print(f"📁 Saved to: {config_path}")

### 9.2 Generate Final Summary

In [ ]:
# Generate final summary
summary_report = f"""
# 🎯 Training Complete - {MODEL_NAME}

## ✅ TensorFlow.js Model Generated Successfully

### 📁 Generated Files in Google Drive:
1. `{MODEL_NAME}_tfjs/` - TensorFlow.js model files ✅
2. `{MODEL_NAME}_config.json` - Configuration for your app ✅

### 📊 Training Summary:
- **Total Training Time**: {total_training_time/60:.1f} minutes
- **Architecture**: MobileNetV2 + Custom Head
- **Parameters**: {best_model.count_params():,}
- **Model Size**: {total_size/1024/1024:.1f} MB
- **Classes**: {len(CLASS_NAMES)}
- **Training Samples**: {len(train_paths):,}

### 🚀 Next Steps:
1. Download model files from Google Drive
2. Upload to your hosting platform
3. Update your models.json configuration
4. Test integration with your application

### 🔄 For Production Training:
- Set `QUICK_TEST_MODE = False`
- Increase `MAX_IMAGES_PER_CLASS` or set to `None`
- Allow 2-4 hours for full training
"""

# Save summary
summary_path = f'/content/drive/MyDrive/{MODEL_NAME}_training_summary.md'
with open(summary_path, 'w') as f:
    f.write(summary_report)

print("🎊 TRAINING COMPLETED SUCCESSFULLY!")
print(f"⏰ Total time: {total_training_time/60:.1f} minutes")
print(f"📱 TensorFlow.js model ready for use!")
print(f"📁 Files saved to Google Drive: {MODEL_NAME}_tfjs/")
print(f"🔗 Configuration ready for integration!")

if QUICK_TEST_MODE:
    print(f"\n🧪 QUICK TEST MODE - Pipeline validated!")
    print("🔄 Set QUICK_TEST_MODE=False for production training")